# Part 1 : Practise

## Task 1: Take a first look

In [ ]:
# load the file and print the shape

%pip install openpyxl
import pandas as pd

listings = pd.read_excel('data/listings.xlsx', engine='openpyxl')
listings.shape

**Answer:** 4574 rows and 84 columns

In [ ]:
# look at the first 5 rows to see what the data looks like
listings.head()

In [ ]:
# summary name of the non-missing count, data type
listings.info()

## Task 2 : Missing values in two columns

In [ ]:
# count values are NOT missing in bedrooms and beds
listings[['bedrooms', 'beds']].count()

In [ ]:
# missing = total rows - not missing
missing = listings.shape[0] - listings[['bedrooms', 'beds']].count()
missing

***Key points *** beds has more missing values: 101 (bedrooms is missing only 2).

## Task 3 : Choose columns

In [ ]:
# view = list of the columns 
# listings[view] keeps only those columns, .head() shows the first 5 rows

view = ["name", "neighbourhood_cleansed", "bedrooms", "beds", "price"]
listings[view].head()

In [ ]:
# Fixed: double brackets, so the inner [ ] is a list of column names
listings[["name", "price"]]

**keypoints:** `listings['name', 'price']` gave a KeyError: ('name', 'price').
With single brackets, Python packed the two names into one tuple, so pandas looked for
one column called ('name', 'price'), which doesn't exist.
Fix: use double brackets, `listings[['name', 'price']]`. The inner brackets make a list of column names.

## Task 4 : Describe the numbers

In [ ]:
listings[["bedrooms", "beds", "availability_365", "minimum_nights"]].describe().round(2)

**Answer:** minimum_nights has the biggest max (500). 500 nights is more than a year,
so it doesn't make sense for a rental. Probably a mistake, or the host just doesn't want bookings.

## Task 5 : Describe the text

In [ ]:
# 4574 listings, 44 different property types. The most common is "Entire rental unit", with 2442 listings.
# 4574 listings, 14 different neighbourhoods. The most common is "Södermalms", with 1145 listings.

listings[["property_type", "neighbourhood_cleansed"]].describe()

In [ ]:
# nunique = number of unique values
listings[["property_type"]].nunique()

In [ ]:
listings[["neighbourhood_cleansed"]].nunique()

**keypoints:**
The dataset contains 4,574 listings spread across 44 unique property types and 14 distinct neighbourhoods.

Property Types: The most prevalent property type is "Entire rental unit", accounting for 2,442 listings.

Neighbourhoods: "Södermalms" is the most common neighbourhood, representing 1,145 listings.

Note: Using .nunique() confirmed these exact counts (44 property types and 14 neighbourhoods), matching the unique row from .describe().

## Task 6 : The price 

In [ ]:
listings["price"].head(3)

In [ ]:
listings[['name', 'price']].sort_values('price', ascending=False).head(3)

**keypoints:**
The price column is currently stored as a string (str) containing currency symbols and commas (e.g., $1,091.00, $702.00). Because of this, Pandas sorts it lexicographically (character by character like a dictionary) rather than numerically.

For example, when sorting from high to low, $999.60 incorrectly appears before $1,091.00 because '9' comes after '1'. To ensure accurate sorting, the price column must first be cleaned (removing $ and ,) and converted into a numeric data type.

## Task 7 : Filter rows

In [ ]:
# (a) listings with 2 or more bedrooms
mask = listings['bedrooms'] >= 2

In [ ]:
mask.sum()

In [ ]:
# share of listings with 2+ bedrooms (mean of True/False = fraction of True)

# 0.399 × 100 = 39.9%  ≈  40%
# mean() gives fraction, and a fraction becomes percent when you multiple by 100
# share of all listings?

mask.mean()

**Answer:** 1825 listings have 2 or more bedrooms. That's about 40% of all listings (mean 0.399).

In [ ]:
# list all column names to find the right ones (room_type, neighbourhood_cleansed)
listings.columns

In [ ]:
# (b) First try with "and" (commented out): it gave ValueError: The truth value of a Series is ambiguous
# listings["neighbourhood_cleansed"] == "Östermalms" and listings["room_type"] == "Entire home/apt" and listings["bedrooms"] >= 2

# Fix: & between the conditions, each condition in ( )
mask = (listings["neighbourhood_cleansed"] == "Östermalms") & (listings["room_type"] == "Entire home/apt") & (listings["bedrooms"] >= 2)
listings[mask].shape

**keypoints:** With `and` I got *ValueError: The truth value of a Series is ambiguous*. `and` needs one True/False, but each condition gives a whole Series of True/False values.
Using `&` with each condition in parentheses compares row by row. 93 listings match (shape (93, 84)).

## Task 8 : Sort and Pick

In [ ]:
mask = (listings["neighbourhood_cleansed"] == "Östermalms")
Östermalms = listings[mask]
Östermalms.shape


In [ ]:
Östermalms.sort_values("number_of_reviews", ascending=False)[["name", "room_type" ,"number_of_reviews"]].head(5)

**Answer:** I kept only the Östermalms listings with a mask (297 listings), sorted them by number_of_reviews from high to low and took the top 5.
The listing at the top is "Great Room in City Center" with 855 reviews, and its room type is **Private room**.


# Part 2 : Go Deeper

## Task 1 : Central or not

In [ ]:
# the four central neighbourhoods
central = ["Södermalms", "Östermalms", "Norrmalms", "Kungsholmens"]

# isin() checks each row: is its neighbourhood in the central list? (True/False per row)
central_mask = listings["neighbourhood_cleansed"].isin(central)

# sum counts the Trues = how many listings are central
central_mask.sum()

In [ ]:
# mean = share of all listings that are central
central_mask.mean()

In [ ]:
# split the listings into two groups
# central_listings: rows where central_mask is True
# outside_listings: ~ flips the mask, so this keeps the rows that are NOT central
central_listings = listings[central_mask]
outside_listings = listings[~central_mask]

In [ ]:
# among the central listings: True if Private room, then mean = share of private rooms
(central_listings["room_type"] == "Private room").mean()

In [ ]:
# same check for the listings outside the centre, to compare with the line above
(outside_listings["room_type"] == "Private room").mean()

**Answer:** 2331 listings are central, which is about 51% of all listings.
Among the central listings about 15% are private rooms (0.151), and outside the centre about 19% (0.194).
So private rooms are more common outside the centre.

## Task 2 : Room for a group

In [ ]:
# True for rows where the room type is Entire home/apt
entire_mask = listings["room_type"] == "Entire home/apt"

In [ ]:
entire_listings = listings[entire_mask]
entire_listings

In [ ]:
other_listings = listings[~entire_mask]
other_listings

In [ ]:
(entire_listings["accommodates"] >= 4).mean()

In [ ]:
(other_listings["accommodates"] >= 4).mean()

**Answer:** About 56% of the Entire home/apt listings have room for 4 or more guests, but only about 9% of the other listings do.

It tells me that a group of 4 or more has many more options among entire homes than among private, shared or hotel rooms.

It does not tell me why, or anything about price or quality. For example, it doesn't show that entire homes are better value or more popular. It only compares the share that fits 4+ guests.

## Task 3 : Best Reviewed, fairly

In [ ]:
# listings that has at least 50 reviews
reviewed_mask = listings["number_of_reviews"] >= 50
reviewed_mask

In [ ]:
# keep only the listings with at least 50 reviews
higher_reviews = listings[reviewed_mask]

# count how many of them have a rating of exactly 5.0
(higher_reviews["review_scores_rating"] == 5.0).sum()

In [ ]:
higher_reviews[["name", "neighbourhood_cleansed", "number_of_reviews", "review_scores_rating"]].sort_values(["review_scores_rating", "number_of_reviews"], ascending=[False,False]).head(5)

In [ ]:
higher_reviews[["name", "neighbourhood_cleansed", "number_of_reviews", "review_scores_rating"]].nlargest(5, "review_scores_rating")

**Answer:** 14 of the listings with at least 50 reviews have a rating of exactly 5.0.

The two top fives are different because those 14 listings are all tied at 5.0, and the top five is picked from among them.
With sort_values I sorted by rating and then by number of reviews, so among the 5.0 listings the one with the most reviews comes first. The top one is "Breakfast paradise City Stockholm" with 159 reviews.
nlargest only looks at review_scores_rating, so it doesn't break the tie and just takes the first five 5.0 listings it finds in the table.

The sort_values list is fairer, because when two listings have the same rating, it puts the one with more reviews first, and more reviews means the rating is more reliable.

## Task 4 : Label or position

In [ ]:
top = higher_reviews[["name", "neighbourhood_cleansed", "number_of_reviews", "review_scores_rating"]].sort_values(["review_scores_rating", "number_of_reviews"], ascending=[False,False]).head(5)
top

In [ ]:
top.loc[347, "name"]

In [ ]:
top.iloc[3]["name"]

In [ ]:
top = top.reset_index(drop=True)

**Answer:** Before the reset, top.loc[347, "name"] and top.iloc[3]["name"] both give "Stockholm", because the row with label 347 is also the 4th row (position 3).

After top.reset_index(drop=True) the old labels are thrown away and the rows get new labels 0 to 4. 
for eg : before:  559, 2115, 3566, 347, ...   →   after:  0, 1, 2, 3, 4

So top.loc[347, "name"] gives a KeyError: 347, because that label doesn't exist anymore.
iloc[3] still works, because it counts the position, and the rows didn't move.

## Task 5 : What does a Zero mean?

In [ ]:
zero_bedrooms = listings["bedrooms"] == 0
zero_bedrooms.sum()

In [ ]:
zero_bedroom_listings = listings[zero_bedrooms]
zero_bedroom_listings["room_type"].describe()


**Answer (a):** 257 listings have 0 bedrooms, and 231 of them are Entire home/apt.
0 bedrooms probably means a studio apartment, where the bed is in the same room as the living area, so there is no separate bedroom.
It could also be a mistake, if the host typed 0 instead of the real number, but most of them being entire homes fits the studio explanation.

In [ ]:
availability_365 = listings["availability_365"] == 0
availability_365.sum()

In [ ]:
availability_365_listings = listings[availability_365]
(availability_365_listings["number_of_reviews"] == 0).sum()

**keypoints (b):** 

A total of 1,494 listings have zero annual availability (availability_365 = 0).

Unreviewed Listings (231): With no reviews on record, these properties likely never hosted guests, making account closure or inactivity the most probable reason rather than complete bookings.

Reviewed Listings (1,263): Because these properties have historical reviews, zero availability could either mean they are fully booked for the upcoming year or were previously active but have since closed/unlisted.

Limitation: Disambiguating fully booked listings from inactive ones would require additional temporal data, such as last_review dates or active calendar availability trends.

## Task 6 : Your own question

**Question:** do bigger listings (4+ guests) get more reviews on average than smaller ones?

In [ ]:
mask_accomodates = listings["accommodates"] >= 4

In [ ]:
accomodates_listings = listings[mask_accomodates]
accomodates_listings.shape

In [ ]:
accomodates_listings["number_of_reviews"].mean()

In [ ]:
accomodates_listings["number_of_reviews"].median()

In [ ]:
less_accomodates_listings = listings[~mask_accomodates]
less_accomodates_listings.shape

In [ ]:
less_accomodates_listings["number_of_reviews"].mean()

In [ ]:
less_accomodates_listings["number_of_reviews"].median()

**keypoints:** Relationship Between Listing Size and Review Count:

The Finding: Smaller listings tend to receive more reviews than larger ones. On average, smaller listings have 39.6 reviews compared to 31.5 for larger listings (median values are 11 and 8, respectively).

Limitations & Unknowns: This data does not reveal the underlying cause—smaller properties might be more affordable or simply listed on Airbnb for a longer duration. Furthermore, because review rates vary (not every guest leaves a review), review counts serve as a proxy rather than an exact measure of guest volume.